# Truthprint — 실증 데이터 핸드오프 (Kaggle · Colab · Local)

**셀을 위에서부터 순서대로 실행**하면 실측 실험용 데이터(실제 번역문 + 사람 주석 초안)를
만들어 **zip으로 내려받는 것**까지 끝냅니다. 이 노트북은 Kaggle·Colab·로컬 Jupyter를
**자동 감지**하므로 어디서 열어도 동작합니다.

> ⚠️ **Kaggle 사용자 필독:** 오른쪽 패널 **Settings → Internet → On** 을 켜세요
> (전화 인증된 계정 필요). 인터넷이 꺼져 있으면 저장소 clone·설치·번역이 실패합니다.
> 그리고 **Settings → Accelerator 는 None(CPU)** 로 두면 됩니다(GPU 불필요).

## STEP 0 — 환경 감지 + 저장소 클론 + 설치

In [ ]:
import os, sys, subprocess

# 작업 베이스 디렉터리 자동 감지 (Kaggle / Colab / 로컬)
if os.path.isdir('/kaggle/working'):
    BASE = '/kaggle/working'
elif os.path.isdir('/content'):
    BASE = '/content'
else:
    BASE = os.getcwd()
REPO = os.path.join(BASE, 'truthprint')
print('BASE =', BASE)
print('REPO =', REPO)

if not os.path.isdir(REPO):
    r = subprocess.run(['git','clone','--depth','1',
                        'https://github.com/leemgs/truthprint', REPO])
    if r.returncode != 0:
        raise RuntimeError('git clone 실패 -> 인터넷이 꺼져 있는지 확인하세요 '
                           '(Kaggle: Settings > Internet > On).')

# 편집가능(editable) 설치
r = subprocess.run([sys.executable,'-m','pip','install','-q','-e',
                    os.path.join(REPO,'code')+'[dev]'])
if r.returncode != 0:
    print('pip install 경고: 계속 진행합니다(테스트에는 sys.path로 충분).')
sys.path.insert(0, os.path.join(REPO,'code'))

# 이후 셀에서 쓸 경로
SRC  = os.path.join(REPO, 'handoff', 'samples')
WORK = os.path.join(BASE, 'my_handoff_data')
assert os.path.isdir(SRC), f'경로 없음: {SRC} (clone 실패로 보임)'
print('OK  SRC =', SRC)
print('OK  WORK will be =', WORK)

## STEP 1 — 코드가 실제로 도는지 확인 (권장)

In [ ]:
from truthprint.cli import main as tp_main
print('--- selftest ---');  tp_main(['selftest'])
print('--- challenge ---'); tp_main(['challenge'])

## STEP 2 — 작업 폴더 만들기 (예제 복사)

`01_source_items.jsonl` 과 `split.json` 은 그대로 둡니다(제가 제공). 나머지를 채웁니다.

In [ ]:
import shutil, os
if os.path.isdir(WORK):
    shutil.rmtree(WORK)
shutil.copytree(SRC, WORK)
# 선택 파일은 빈 템플릿으로 초기화(예제 ID가 자동 번역 결과와 안 맞아 생기는 경고 방지)
with open(os.path.join(WORK,'04_human_factuality.csv'),'w',encoding='utf-8') as fh:
    fh.write('pair_id,doc_id,sent_id,transform_id,kind,field_if_altering,human_equivalent,annotator_id,notes\n')
open(os.path.join(WORK,'05_baseline_outputs.jsonl'),'w',encoding='utf-8').close()
print('working folder:', WORK)
print(sorted(os.listdir(WORK)))

## STEP 3 — 번역할 원문 문장 목록 뽑기

In [ ]:
import json, csv
rows = []
with open(os.path.join(WORK,'01_source_items.jsonl'), encoding='utf-8') as fh:
    for line in fh:
        r = json.loads(line)
        for sid, text in r['watermarked_text'].items():
            rows.append((r['doc_id'], sid, text))
with open(os.path.join(WORK,'to_translate.csv'),'w',newline='',encoding='utf-8') as fh:
    w = csv.writer(fh); w.writerow(['doc_id','sent_id','source_text']); w.writerows(rows)
print(f'{len(rows)} sentences -> to_translate.csv')
for _, sid, t in rows:
    print(' ', sid, '|', t)

## STEP 4 — 실제 번역 만들기  ⭐

무료 Google 번역(`deep-translator`, API 키 불필요)으로 EN→KO, EN→HI, round-trip(EN→KO→EN)을
실제 수행해 `02_transformations.jsonl` 을 채웁니다. (인터넷 필요)

- 다른 시스템(DeepL/NLLB/GPT/Claude 등)을 쓰려면 이 셀 대신 그 출력으로 같은 형식의 파일을
  만들고 `system` 값을 실제 이름으로 바꾸면 됩니다.

In [ ]:
import subprocess, sys, json, os, time
subprocess.run([sys.executable,'-m','pip','install','-q','deep-translator'])
from deep_translator import GoogleTranslator

SYSTEM = 'deep-translator/GoogleTranslator(web)'
THROTTLE = 1.0   # 요청 간 지연(초). 429가 나면 2.0~3.0으로 올리세요.

def tr(text, src, tgt, tries=6):
    '''속도제한(429) 대비: 지수 백오프로 재시도.'''
    for i in range(tries):
        try:
            time.sleep(THROTTLE)
            return GoogleTranslator(source=src, target=tgt).translate(text)
        except Exception as e:
            wait = min(30, 2 ** i)
            print(f'    재시도 {i+1}/{tries} ({type(e).__name__}) -> {wait}s 대기')
            time.sleep(wait)
    raise RuntimeError('번역 재시도 소진(속도제한 지속). THROTTLE를 올리거나 잠시 후 다시 실행하세요.')

out, failed = [], []
for doc_id, sid, text in rows:
    try:
        ko = tr(text,'en','ko'); hi = tr(text,'en','hi'); rt = tr(ko,'ko','en')
    except Exception as e:
        print(f'  [skip] {sid}: {e}'); failed.append(sid); continue
    out += [
      {'transform_id':f'{sid}-ko','doc_id':doc_id,'sent_id':sid,'transform_type':'translation','direction':'en->ko','system':SYSTEM,'params':{},'output_text':ko,'round_trip':False},
      {'transform_id':f'{sid}-hi','doc_id':doc_id,'sent_id':sid,'transform_type':'translation','direction':'en->hi','system':SYSTEM,'params':{},'output_text':hi,'round_trip':False},
      {'transform_id':f'{sid}-rt','doc_id':doc_id,'sent_id':sid,'transform_type':'roundtrip_translation','direction':'en->ko->en','system':SYSTEM,'params':{},'output_text':rt,'round_trip':True},
    ]
    print(f'  ok {sid}')

with open(os.path.join(WORK,'02_transformations.jsonl'),'w',encoding='utf-8') as fh:
    for r in out:
        fh.write(json.dumps(r, ensure_ascii=False)+'\n')
print(f'\nwrote {len(out)} transformations for {len(rows)-len(failed)}/{len(rows)} sentences')
if failed:
    print('실패(속도제한 등):', failed, '-> THROTTLE 올리고 이 셀만 다시 실행하면 이어서 채워집니다.')
for r in out[:6]:
    print(' ', r['transform_id'], '|', r['output_text'])

## STEP 5 — 사람 주석 (초안 자동 생성 → 당신이 검토·수정)  ⭐

아래 셀은 **초안** `03_annotations.jsonl` 을 만듭니다(번역이 의미를 보존했다고 가정하고
원문 불변량을 채움). **번역문을 읽고 반드시 검토**하세요:

1. 의미(극성·수량·시간방향·양태·귀속·인과)가 바뀌었으면 해당 필드 수정 + `invariant_preserved=false`.
2. 태(voice)/시간구 위치 carrier가 사라졌으면 그 carrier `reliable=false`(→ 검출에서 erasure).

필드 정의: 리포의 `handoff/schemas/SCHEMA_KO.md`.

In [ ]:
import json, os
src_inv = {}
with open(os.path.join(WORK,'01_source_items.jsonl'), encoding='utf-8') as fh:
    for line in fh:
        r = json.loads(line)
        for fct in r['facts']:
            src_inv[fct['sent_id']] = {k: fct[k] for k in ['agent','patient','predicate',
                'quantity','polarity','time_dir','modality','attribution','causation']}

draft = []
with open(os.path.join(WORK,'02_transformations.jsonl'), encoding='utf-8') as fh:
    for line in fh:
        t = json.loads(line); sid = t['sent_id']
        draft.append({'transform_id':t['transform_id'],'annotator_id':'A1_DRAFT',
            'invariants_observed':dict(src_inv[sid]),
            'carriers_observed':[{'carrier':'voice','value':'active','reliable':True},
                                 {'carrier':'time_position','value':'front','reliable':True}],
            'invariant_preserved':True,
            'notes':'AUTO-DRAFT: 번역문 검토 후 수정하세요. 의미변경 시 필드+invariant_preserved=false; carrier 소거 시 reliable=false.'})
with open(os.path.join(WORK,'03_annotations.jsonl'),'w',encoding='utf-8') as fh:
    for r in draft:
        fh.write(json.dumps(r, ensure_ascii=False)+'\n')
print(f'wrote {len(draft)} DRAFT annotations -> 검토 필요')

## STEP 6 — (선택) 사람 의미동일성 / baseline
`04_human_factuality.csv`, `05_baseline_outputs.jsonl` 은 예제 형식대로 채우면 됩니다.
어려우면 건너뛰어도 검증은 통과합니다.

## STEP 7 — 검증기로 형식 점검 (READY 뜰 때까지)

In [ ]:
import subprocess, sys, os
subprocess.run([sys.executable, os.path.join(REPO,'handoff','validate_handoff.py'), WORK])

## STEP 8 — 결과 zip 만들기 → 나에게 전달

아래 셀이 `my_handoff_data.zip` 을 만듭니다.
- **Kaggle:** 오른쪽 **Output** 패널(또는 `/kaggle/working`)에서 zip을 다운로드하세요.
  (Save Version 후 Output 탭에서도 받을 수 있습니다.)
- **Colab:** 자동 다운로드가 뜹니다.

그 파일을 저에게 주시거나, 리포 브랜치에 올린 뒤 이 세션에 알려주세요:
```
my_handoff_data 채웠고 validate READY 떴어. 실측 실험 돌려서 논문 표 채워줘.
```

In [ ]:
import shutil, os
zip_base = os.path.join(BASE,'my_handoff_data')
zip_path = shutil.make_archive(zip_base, 'zip', WORK)
print('created:', zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print('Kaggle/로컬: 위 경로의 zip 파일을 직접 다운로드하세요.')
    print('Kaggle이면 오른쪽 Output 패널에 my_handoff_data.zip 이 보입니다.')